In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
data=pd.read_csv("Diabetes_Prediction_Dataset.csv")

In [3]:
data.shape

(5000, 13)

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Patient_ID         5000 non-null   int64  
 1   Age                5000 non-null   int64  
 2   Gender             5000 non-null   str    
 3   BMI                4893 non-null   float64
 4   Glucose            4930 non-null   float64
 5   Blood_Pressure     5000 non-null   int64  
 6   Cholesterol        5000 non-null   int64  
 7   Insulin            5000 non-null   int64  
 8   HbA1c              5000 non-null   float64
 9   Smoking_Status     5000 non-null   str    
 10  Physical_Activity  5000 non-null   str    
 11  Family_History     5000 non-null   str    
 12  Diabetes           5000 non-null   str    
dtypes: float64(3), int64(5), str(5)
memory usage: 611.4 KB


In [5]:
data.describe()

,Patient_ID,Age,BMI,Glucose,Blood_Pressure,Cholesterol,Insulin,HbA1c
count,5000.000000,5000.000000,4893.000000,4930.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2500.500000,48.986400,29.407398,144.456795,143.139200,221.426800,157.746200,7.988300
std,1443.520003,18.173813,7.254359,43.887777,87.414816,57.436379,82.357275,2.017408
min,1.000000,18.000000,17.000000,70.000000,90.000000,120.000000,15.000000,4.500000
25%,1250.750000,33.000000,23.100000,106.000000,112.000000,172.000000,86.000000,6.200000
50%,2500.500000,49.000000,29.400000,144.000000,136.000000,222.000000,158.000000,7.900000
75%,3750.250000,65.000000,35.800000,182.000000,158.000000,271.000000,229.000000,9.800000
max,5000.000000,80.000000,42.000000,220.000000,999.000000,320.000000,300.000000,11.500000


In [6]:
data.isnull().sum()

Patient_ID             0
Age                    0
Gender                 0
BMI                  107
Glucose               70
Blood_Pressure         0
Cholesterol            0
Insulin                0
HbA1c                  0
Smoking_Status         0
Physical_Activity      0
Family_History         0
Diabetes               0
dtype: int64

In [7]:
data["BMI"] = data["BMI"].fillna(data["BMI"].median())
data["Glucose"] = data["Glucose"].fillna(data["Glucose"].median())

In [8]:
data.isnull().sum()

Patient_ID           0
Age                  0
Gender               0
BMI                  0
Glucose              0
Blood_Pressure       0
Cholesterol          0
Insulin              0
HbA1c                0
Smoking_Status       0
Physical_Activity    0
Family_History       0
Diabetes             0
dtype: int64

In [9]:
print(data["Gender"].unique())
print(data["Smoking_Status"].unique())
print(data["Physical_Activity"].unique())
print(data["Family_History"].unique())

<ArrowStringArray>
['Male', 'Female', 'Unknown']
Length: 3, dtype: str
<ArrowStringArray>
['Current', 'Never', 'Former']
Length: 3, dtype: str
<ArrowStringArray>
['High', 'Low', 'Moderate']
Length: 3, dtype: str
<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


In [10]:
print((data["Gender"] == "Unknown").sum())

46


In [11]:
data = data[data["Gender"] != "Unknown"]

In [12]:
print(data["Gender"].unique())

<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str


In [13]:
data.head()

,Patient_ID,Age,Gender,BMI,Glucose,Blood_Pressure,Cholesterol,Insulin,HbA1c,Smoking_Status,Physical_Activity,Family_History,Diabetes
0,1,58,Male,17.6,140.0,121,177,86,9.7,Current,High,Yes,Yes
1,2,56,Male,31.0,209.0,143,176,244,8.6,Never,Low,No,Yes
2,3,23,Female,19.4,158.0,167,187,37,9.6,Current,Low,No,No
3,4,41,Male,34.6,81.0,174,178,163,11.4,Never,Low,No,No
4,5,31,Female,34.5,88.0,167,282,102,8.2,Never,Low,No,No


In [14]:
data.describe()

,Patient_ID,Age,BMI,Glucose,Blood_Pressure,Cholesterol,Insulin,HbA1c
count,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000
mean,2501.600525,48.962858,29.405067,144.445095,143.121518,221.360920,157.792693,7.986536
std,1442.647043,18.186103,7.166727,43.582452,86.934119,57.465299,82.341400,2.017165
min,1.000000,18.000000,17.000000,70.000000,90.000000,120.000000,15.000000,4.500000
25%,1250.250000,33.000000,23.200000,107.000000,112.000000,172.000000,87.000000,6.200000
50%,2504.500000,49.000000,29.400000,144.000000,136.000000,222.000000,158.000000,7.900000
75%,3749.750000,65.000000,35.600000,182.000000,159.000000,271.000000,229.000000,9.775000
max,5000.000000,80.000000,42.000000,220.000000,999.000000,320.000000,300.000000,11.500000


In [15]:
# Find unrealistic values
print(data[data["Blood_Pressure"] > 250])


      Patient_ID  Age  Gender   BMI  Glucose  Blood_Pressure  Cholesterol  \
36            37   22  Female  27.2    208.0             999          226   
67            68   29    Male  21.4    167.0             999          294   
226          227   54    Male  31.1    118.0             999          258   
286          287   52  Female  19.3    126.0             999          289   
290          291   29  Female  31.0    217.0             999          186   
355          356   72  Female  27.3    180.0             999          275   
424          425   54  Female  36.4     74.0             999          158   
432          433   62  Female  30.8    182.0             999          268   
556          557   39  Female  29.4    185.0             999          182   
585          586   30  Female  36.3    176.0             999          232   
705          706   28    Male  25.7    177.0             999          241   
741          742   62  Female  26.8     95.0             999          307   

In [16]:
# Replace them with the median
data.loc[data["Blood_Pressure"] > 250, "Blood_Pressure"] = data["Blood_Pressure"].median()

In [17]:
data.describe()

,Patient_ID,Age,BMI,Glucose,Blood_Pressure,Cholesterol,Insulin,HbA1c
count,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000,4954.000000
mean,2501.600525,48.962858,29.405067,144.445095,135.108195,221.360920,157.792693,7.986536
std,1442.647043,18.186103,7.166727,43.582452,26.278855,57.465299,82.341400,2.017165
min,1.000000,18.000000,17.000000,70.000000,90.000000,120.000000,15.000000,4.500000
25%,1250.250000,33.000000,23.200000,107.000000,112.000000,172.000000,87.000000,6.200000
50%,2504.500000,49.000000,29.400000,144.000000,136.000000,222.000000,158.000000,7.900000
75%,3749.750000,65.000000,35.600000,182.000000,158.000000,271.000000,229.000000,9.775000
max,5000.000000,80.000000,42.000000,220.000000,180.000000,320.000000,300.000000,11.500000


In [18]:
#Count duplicate rows-
print(data.duplicated().sum())

0


In [19]:
print(data[data.duplicated()])

Empty DataFrame
Columns: [Patient_ID, Age, Gender, BMI, Glucose, Blood_Pressure, Cholesterol, Insulin, HbA1c, Smoking_Status, Physical_Activity, Family_History, Diabetes]
Index: []


In [20]:
data=data.drop_duplicates()

In [21]:
from sklearn.preprocessing import LabelEncoder

In [22]:
le = LabelEncoder()

In [23]:
data["Gender"] = le.fit_transform(data["Gender"])

data["Smoking_Status"] = le.fit_transform(data["Smoking_Status"])

data["Physical_Activity"] = le.fit_transform(data["Physical_Activity"])

data["Family_History"] = le.fit_transform(data["Family_History"])

data["Diabetes"] = le.fit_transform(data["Diabetes"])

In [24]:
data.head()

,Patient_ID,Age,Gender,BMI,Glucose,Blood_Pressure,Cholesterol,Insulin,HbA1c,Smoking_Status,Physical_Activity,Family_History,Diabetes
0,1,58,1,17.6,140.0,121,177,86,9.7,0,0,1,1
1,2,56,1,31.0,209.0,143,176,244,8.6,2,1,0,1
2,3,23,0,19.4,158.0,167,187,37,9.6,0,1,0,0
3,4,41,1,34.6,81.0,174,178,163,11.4,2,1,0,0
4,5,31,0,34.5,88.0,167,282,102,8.2,2,1,0,0


In [25]:
print(data.dtypes)

Patient_ID             int64
Age                    int64
Gender                 int64
BMI                  float64
Glucose              float64
Blood_Pressure         int64
Cholesterol            int64
Insulin                int64
HbA1c                float64
Smoking_Status         int64
Physical_Activity      int64
Family_History         int64
Diabetes               int64
dtype: object


In [26]:
data = data.drop("Patient_ID", axis=1)

In [27]:
# Features (Input)
X = data.drop("Diabetes", axis=1)

# Target (Output)
y = data["Diabetes"]

In [28]:
X.head()

,Age,Gender,BMI,Glucose,Blood_Pressure,Cholesterol,Insulin,HbA1c,Smoking_Status,Physical_Activity,Family_History
0,58,1,17.6,140.0,121,177,86,9.7,0,0,1
1,56,1,31.0,209.0,143,176,244,8.6,2,1,0
2,23,0,19.4,158.0,167,187,37,9.6,0,1,0
3,41,1,34.6,81.0,174,178,163,11.4,2,1,0
4,31,0,34.5,88.0,167,282,102,8.2,2,1,0


In [29]:
y.head()

0    1
1    1
2    0
3    0
4    0
Name: Diabetes, dtype: int64

In [30]:
X.shape

(4954, 11)

In [31]:
y.shape

(4954,)

In [32]:
from sklearn.model_selection import train_test_split

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [34]:
from sklearn.tree import DecisionTreeClassifier

In [35]:
model = DecisionTreeClassifier(random_state=42)

In [36]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [37]:
y_pred = model.predict(X_test)

In [38]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9848637739656912


In [39]:
from sklearn.metrics import precision_score

print("Precision:", precision_score(y_test, y_pred))

Precision: 0.9833610648918469


In [40]:
from sklearn.metrics import recall_score

print("Recall:", recall_score(y_test, y_pred))

Recall: 0.9916107382550335


In [41]:
from sklearn.metrics import f1_score

print("F1 Score:", f1_score(y_test, y_pred))

F1 Score: 0.9874686716791979


In [42]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[385  10]
 [  5 591]]


In [43]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']

In [44]:
model = joblib.load("model.pkl")